<a href="https://colab.research.google.com/github/Mennaali30/recipe-agent/blob/main/notebooks/run_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/Mennaali30/recipe-agent.git
%cd recipe-agent

Cloning into 'recipe-agent'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 55 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 19.66 KiB | 491.00 KiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/recipe-agent


In [3]:
%pip install -q -r requirements.txt kagglehub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [6]:
import os, sys, shutil
from google.colab import drive

drive.mount('/content/drive')
A = "/content/drive/MyDrive/recipe-agent-artifacts"
os.makedirs(A, exist_ok=True)
sys.path.append("/content/recipe-agent/src")
PARQUET = "/content/recipe-agent/data/processed/recipes.parquet"

if os.path.exists(f"{A}/recipes.parquet"):
    os.makedirs(os.path.dirname(PARQUET), exist_ok=True)
    shutil.copy(f"{A}/recipes.parquet", PARQUET)
    print("restored recipes.parquet from Drive")
else:
    import kagglehub
    from recipe_agent.ingest import run
    path = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews")
    run(raw_dir=path)
    shutil.copy(PARQUET, f"{A}/recipes.parquet")
    print("built and saved recipes.parquet")

Mounted at /content/drive
restored recipes.parquet from Drive


In [11]:
from recipe_agent.retriever import build_index

if os.path.exists(f"{A}/chroma_db.zip"):
    shutil.unpack_archive(f"{A}/chroma_db.zip", "/content/recipe-agent")
    print("restored chroma_db from Drive")
else:
    print("total indexed:", build_index())
    shutil.make_archive(f"{A}/chroma_db", "zip", root_dir="/content/recipe-agent", base_dir="chroma_db")
    print("saved chroma_db.zip to Drive")

print(os.listdir(A))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

indexed 256/19,554
indexed 512/19,554
indexed 768/19,554
indexed 1,024/19,554
indexed 1,280/19,554
indexed 1,536/19,554
indexed 1,792/19,554
indexed 2,048/19,554
indexed 2,304/19,554
indexed 2,560/19,554
indexed 2,816/19,554
indexed 3,072/19,554
indexed 3,328/19,554
indexed 3,584/19,554
indexed 3,840/19,554
indexed 4,096/19,554
indexed 4,352/19,554
indexed 4,608/19,554
indexed 4,864/19,554
indexed 5,120/19,554
indexed 5,376/19,554
indexed 5,632/19,554
indexed 5,888/19,554
indexed 6,144/19,554
indexed 6,400/19,554
indexed 6,656/19,554
indexed 6,912/19,554
indexed 7,168/19,554
indexed 7,424/19,554
indexed 7,680/19,554
indexed 7,936/19,554
indexed 8,192/19,554
indexed 8,448/19,554
indexed 8,704/19,554
indexed 8,960/19,554
indexed 9,216/19,554
indexed 9,472/19,554
indexed 9,728/19,554
indexed 9,984/19,554
indexed 10,240/19,554
indexed 10,496/19,554
indexed 10,752/19,554
indexed 11,008/19,554
indexed 11,264/19,554
indexed 11,520/19,554
indexed 11,776/19,554
indexed 12,032/19,554
indexed 12,

In [13]:
import sys
sys.path.append("/content/recipe-agent/src")
from recipe_agent.retriever import search

for q, mins in [("Ingredients: chicken, rice, garlic", None), ("Ingredients: chicken, rice, garlic", 30)]:
    print(f"\n>>> {q} | max_minutes={mins}")
    for h in search(q, n_results=5, max_minutes=mins):
        print(f"  {h['name']} | {h['category']} | {h['total_minutes']} min | {h['calories']} cal | dist={h['distance']}")


>>> Ingredients: chicken, rice, garlic | max_minutes=None
  Chicken and Rice | Chicken | 30 min | 316.2 cal | dist=0.1564
  Chicken Tikka Masala by Jamie Oliver | Curries | 75 min | 470.6 cal | dist=0.1571
  Garlic Chicken on Rice | Chicken Breast | 30 min | 332.6 cal | dist=0.1601
  Chicken Rice Provencal | Chicken | 60 min | 346.3 cal | dist=0.1652
  Italian Chicken & Rice for Two | Chicken Breast | 30 min | 600.9 cal | dist=0.1687

>>> Ingredients: chicken, rice, garlic | max_minutes=30
  Chicken and Rice | Chicken | 30 min | 316.2 cal | dist=0.1564
  Garlic Chicken on Rice | Chicken Breast | 30 min | 332.6 cal | dist=0.1601
  Italian Chicken & Rice for Two | Chicken Breast | 30 min | 600.9 cal | dist=0.1687
  Rosemary Chicken With Wild Rice (Weight Watchers) | One Dish Meal | 30 min | 236.9 cal | dist=0.171
  Curried Chicken And Rice Soup | Chicken | 30 min | 381.8 cal | dist=0.172


In [14]:
%cd /content/recipe-agent
!git pull
!ls src/recipe_agent

/content/recipe-agent
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.55 KiB | 1.28 MiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   8058de2..257b335  main       -> origin/main
Updating 8058de2..257b335
Fast-forward
 src/recipe_agent/tools.py | 109 ++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 109 insertions(+)
 create mode 100644 src/recipe_agent/tools.py
config.py  __init__.py	retriever.py  utils.py
ingest.py  __pycache__	tools.py


In [15]:
import sys
sys.path.append("/content/recipe-agent/src")
from recipe_agent.tools import compare_ingredients

print(compare_ingredients(["tomatoes", "eggs"], ["tomato", "egg", "salt", "fresh basil"]))

{'have': ['tomato', 'egg'], 'missing': ['fresh basil'], 'coverage': 0.67}


In [16]:
from recipe_agent.tools import find_recipes, shopping_list

results = find_recipes(["chicken", "rice", "garlic", "lemon"], max_minutes=40)
for r in results:
    print(f"{r['name']} | {r['total_minutes']} min | coverage={r['coverage']} | {r['nutrition']['calories']} cal/serving")
    print("   have:", r["have"])
    print("   missing:", r["missing"])

print("\nshopping list:", shopping_list([r["missing"] for r in results[:2]]))

Najla's Lemon Chicken | 30 min | coverage=0.67 | 341 cal/serving
   have: ['garlic clove', 'boneless skinless chicken breast halves', 'chicken bouillon cube', 'lemon']
   missing: ['butter', 'onion']
Quick Garlic Roast Chicken With Rosemary & Lemon | 35 min | coverage=0.57 | 437 cal/serving
   have: ['boneless skinless chicken breasts', 'garlic cloves', 'lemon, juice and zest of', 'chicken broth']
   missing: ['fresh rosemary', 'extra virgin olive oil', 'dry white wine']
Lemony Chicken Soup | 30 min | coverage=0.56 | 111 cal/serving
   have: ['chicken', 'chicken broth', 'chicken bouillon cubes', 'rice', 'lemon juice']
   missing: ['carrots', 'onion', 'eggs', 'oregano']
Zankou Chicken's Lebanese Garlic Sauce (Copycat) | 25 min | coverage=0.5 | 146 cal/serving
   have: ['garlic', 'fresh lemon juice']
   missing: ['russet potatoes', 'canola oil']
Lemon Teriyaki Chicken | 30 min | coverage=0.5 | 518 cal/serving
   have: ['lemon juice', 'garlic', 'chicken thighs']
   missing: ['soy sauce', 

In [17]:
%cd /content/recipe-agent
!git pull

/content/recipe-agent
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.55 KiB | 396.00 KiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   257b335..5965d5c  main       -> origin/main
Updating 257b335..5965d5c
Fast-forward
 src/recipe_agent/tools.py | 36 ++++++++++++++++++++++++++++++------
 1 file changed, 30 insertions(+), 6 deletions(-)


In [18]:
import sys, importlib
sys.path.append("/content/recipe-agent/src")

import recipe_agent.tools as tools
importlib.reload(tools)

print(tools.compare_ingredients(
    ["chicken", "lemon"],
    ["chicken breast", "chicken broth", "chicken bouillon cubes",
     "lemon juice", "extra virgin olive oil", "canola oil"],
))
print(tools.compare_ingredients(["garlic", "tomato"], ["garlic powder", "garlic cloves", "tomato sauce", "tomatoes"]))

{'have': ['chicken breast', 'lemon juice'], 'missing': ['chicken broth', 'chicken bouillon cubes'], 'coverage': 0.5}
{'have': ['garlic cloves', 'tomatoes'], 'missing': ['garlic powder', 'tomato sauce'], 'coverage': 0.5}


In [19]:
results = tools.find_recipes(["chicken", "rice", "garlic", "lemon"], max_minutes=40)
for r in results:
    print(f"{r['name']} | {r['total_minutes']} min | coverage={r['coverage']} | {r['nutrition']['calories']} cal/serving")
    print("   have:", r["have"])
    print("   missing:", r["missing"])

Zankou Chicken's Lebanese Garlic Sauce (Copycat) | 25 min | coverage=0.67 | 146 cal/serving
   have: ['garlic', 'fresh lemon juice']
   missing: ['russet potatoes']
Lemon Teriyaki Chicken | 30 min | coverage=0.5 | 518 cal/serving
   have: ['lemon juice', 'garlic', 'chicken thighs']
   missing: ['soy sauce', 'sugar', 'brown sugar']
Najla's Lemon Chicken | 30 min | coverage=0.5 | 341 cal/serving
   have: ['garlic clove', 'boneless skinless chicken breast halves', 'lemon']
   missing: ['butter', 'onion', 'chicken bouillon cube']
Quick Garlic Roast Chicken With Rosemary & Lemon | 35 min | coverage=0.5 | 437 cal/serving
   have: ['boneless skinless chicken breasts', 'garlic cloves', 'lemon, juice and zest of']
   missing: ['fresh rosemary', 'dry white wine', 'chicken broth']
Zesty Lemon Pepper Chicken | 40 min | coverage=0.5 | 252 cal/serving
   have: ['chicken breast halves', 'chicken pieces', 'lemons', 'garlic']
   missing: ['onion', 'paprika', 'dried thyme', 'fresh ground pepper']


In [20]:
%cd /content/recipe-agent
!git pull

/content/recipe-agent
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.43 KiB | 730.00 KiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   5965d5c..be495ef  main       -> origin/main
Updating 5965d5c..be495ef
Fast-forward
 src/recipe_agent/tools.py | 26 ++++++++++++++++++++------
 1 file changed, 20 insertions(+), 6 deletions(-)


In [21]:
import sys, importlib
sys.path.append("/content/recipe-agent/src")

import recipe_agent.tools as tools
importlib.reload(tools)

print(tools.compare_ingredients(["chicken"], ["fresh ground pepper", "dried thyme", "chicken breast"]))

results = tools.find_recipes(["chicken", "rice", "garlic", "lemon"], max_minutes=40)
for r in results:
    print(f"{r['name']} | {r['total_minutes']} min | uses {r['used']}/4 of yours | coverage={r['coverage']} | {r['nutrition']['calories']} cal")
    print("   have:", r["have"])
    print("   missing:", r["missing"])

{'have': ['chicken breast'], 'missing': ['dried thyme'], 'coverage': 0.5}
Zesty Lemon Pepper Chicken | 40 min | uses 3/4 of yours | coverage=0.57 | 252 cal
   have: ['chicken breast halves', 'chicken pieces', 'lemons', 'garlic']
   missing: ['onion', 'paprika', 'dried thyme']
Lemon Teriyaki Chicken | 30 min | uses 3/4 of yours | coverage=0.5 | 518 cal
   have: ['lemon juice', 'garlic', 'chicken thighs']
   missing: ['soy sauce', 'sugar', 'brown sugar']
Najla's Lemon Chicken | 30 min | uses 3/4 of yours | coverage=0.5 | 341 cal
   have: ['garlic clove', 'boneless skinless chicken breast halves', 'lemon']
   missing: ['butter', 'onion', 'chicken bouillon cube']
Quick Garlic Roast Chicken With Rosemary & Lemon | 35 min | uses 3/4 of yours | coverage=0.5 | 437 cal
   have: ['boneless skinless chicken breasts', 'garlic cloves', 'lemon, juice and zest of']
   missing: ['fresh rosemary', 'dry white wine', 'chicken broth']
Garlic Chicken on Rice | 30 min | uses 3/4 of yours | coverage=0.43 | 3

In [22]:
%cd /content/recipe-agent
!git pull
!ls src/recipe_agent

/content/recipe-agent
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 15 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 4.70 KiB | 602.00 KiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   be495ef..bdd9d7b  main       -> origin/main
Updating be495ef..bdd9d7b
Fast-forward
 src/recipe_agent/agents.py  | 38 ++++++++++++++++++++++++++++++++++++++
 src/recipe_agent/config.py  |  9 +++++++--
 src/recipe_agent/schemas.py | 25 +++++++++++++++++++++++++
 3 files changed, 70 insertions(+), 2 deletions(-)
 create mode 100644 src/recipe_agent/agents.py
 create mode 100644 src/recipe_agent/schemas.py
agents.py  ingest.py	__pycache__   schemas.py  utils.py
config.py  __init__.py	retriever.py  tools.py


In [23]:
import os, sys
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"
sys.path.append("/content/recipe-agent/src")

In [24]:
from recipe_agent.agents import parse_request

tests = [
    "عندي فراخ وأرز وثوم وعايزة أكلة تخلص في نص ساعة",
    "I have eggs, tomatoes and cheese. No onions please.",
    "عندي بطاطس وبيض وعندي حساسية من اللبن",
]
for t in tests:
    print(t)
    print("  ->", parse_request(t).model_dump(), "\n")

عندي فراخ وأرز وثوم وعايزة أكلة تخلص في نص ساعة


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  -> {'ingredients': ['chicken', 'rice', 'garlic'], 'exclude_ingredients': [], 'max_minutes': 30, 'language': 'ar'} 

I have eggs, tomatoes and cheese. No onions please.
  -> {'ingredients': ['egg', 'tomato', 'cheese'], 'exclude_ingredients': ['onion'], 'max_minutes': None, 'language': 'en'} 

عندي بطاطس وبيض وعندي حساسية من اللبن
  -> {'ingredients': ['potato', 'egg'], 'exclude_ingredients': ['milk'], 'max_minutes': None, 'language': 'ar'} 



In [25]:
%cd /content/recipe-agent
!git pull

/content/recipe-agent
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 4.15 KiB | 1.04 MiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   bdd9d7b..98ba326  main       -> origin/main
Updating bdd9d7b..98ba326
Fast-forward
 src/recipe_agent/agents.py | 37 +++++++++++++++++++++++++++++++++++++
 src/recipe_agent/tools.py  | 21 ++++++++++++++++++++-
 2 files changed, 57 insertions(+), 1 deletion(-)


In [26]:
import os, sys, importlib
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"
sys.path.append("/content/recipe-agent/src")

import recipe_agent.tools as tools
import recipe_agent.agents as agents
importlib.reload(tools)
importlib.reload(agents)

def ask(text, top_k=3):
    req = agents.parse_request(text)
    found = tools.find_recipes(
        req.ingredients, max_minutes=req.max_minutes,
        exclude=req.exclude_ingredients, top_k=top_k,
    )
    for r in found:
        r["instructions"] = tools.get_instructions(r["recipe_id"])
    shop = tools.shopping_list([r["missing"] for r in found])
    print(req.model_dump(), "\n")
    print(agents.write_answer(req, found, shop))

ask("عندي فراخ وأرز وثوم وعايزة أكلة تخلص في نص ساعة")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'ingredients': ['chicken', 'rice', 'garlic'], 'exclude_ingredients': [], 'max_minutes': 30, 'language': 'ar'} 

إليك الوصفات التي يمكنك تحضيرها بناءً على المكونات والوقت المحدد:

### دجاج في المقلاة مع الفاصوليا السوداء والأرز (Skillet Chicken With Black Beans and Rice)

- **الوقت الإجمالي:** 28 دقيقة
- **عدد الوجبات:** 4
- **السعرات الحرارية:** 599 سعرة حرارية لكل وجبة
- **البروتين:** 27.9 غم لكل وجبة
- **المكونات المتوفرة لديك:** صدور دجاج، فصوص ثوم، أرز بني
- **المكونات الناقصة:** فاصوليا سوداء معلبة، صلصة سالسا، ذرة مجمدة، ملح طعام

**الخطوات:**
1. قطّع صدور الدجاج بزاويّة للحصول على شرائح بسمك 1/2 إلى 3/4 بوصة.
2. سخّن زيت الزيتون في المقلاة على حرارة متوسطة إلى عالية.
3. عندما يبدأ زيت الزيتون في "التدخين"، خفّف الحرارة إلى متوسطة وأضف الثوم.
4. قلّب بسرعة حتى تفوح رائحة الثوم فقط (15-20 ثانية) ثم أضف الدجاج.
5. استمر في الطهي على حرارة متوسطة حتى ينضج الدجاج تمامًا ويتحمر قليلاً.
6. أخرج الدجاج من المقلاة وضعه جانبًا.
7. أضف 1 ملعقة كبيرة من الماء إلى المقلاة واكشط القاع حتى تت

In [27]:
ask("عندي بطاطس وبيض وعندي حساسية من اللبن")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'ingredients': ['potato', 'egg'], 'exclude_ingredients': ['milk'], 'max_minutes': None, 'language': 'ar'} 

ملاحظة: تصفية المكونات تعتمد على أسماء المكونات فقط، يُرجى التحقق من الملصقات الغذائية للتأكد من عدم وجود مسببات الحساسية.

---

### نيوشي (Gnocchi)

- **الوقت الإجمالي:** 60 دقيقة
- **عدد الوجبات:** 4
- **السعرات الحرارية:** 327 سعرة حرارية لكل وجبة
- **البروتين:** 10.2 غرام لكل وجبة
- **المكونات المتوفرة لديك:** بطاطس (potatoes)، بيض (egg)
- **المكونات الناقصة:** طحين لجميع الأغراض (all-purpose flour)

**خطوات التحضير:**
1. اغلي قدراً كبيراً من الماء المملح. قشر البطاطس وأضفها إلى القدر. اطبخها حتى تصبح طرية ولكن تظل متماسكة، حوالي 15 دقيقة. صفها، ودعها تبرد ثم اهرسها بشوكة أو بهراسة البطاطس.
2. اخلط كوباً واحداً من البطاطس المهروسة، والطحين والبيضة في وعاء كبير. اعجن حتى تتكون العجينة على شكل كرة. شكل أجزاء صغيرة من العجينة على هيئة "أفاعٍ" طويلة. على سطح مرشوش بالطبين، قطع الأفاعي إلى قطع بحجم نصف بوصة.
3. اغلي قدراً كبيراً من الماء المملح خفيفاً. ضع قطع النيوكي واطبخها لمدة

In [28]:
%cd /content/recipe-agent
!git pull

/content/recipe-agent
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 3.65 KiB | 934.00 KiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   98ba326..9c77f71  main       -> origin/main
Updating 98ba326..9c77f71
Fast-forward
 src/recipe_agent/agents.py |  8 +++++-
 src/recipe_agent/tools.py  | 70 ++++++++++++++++++++++++++++++++++++++++------
 2 files changed, 68 insertions(+), 10 deletions(-)


In [29]:
import os, sys, importlib
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"
sys.path.append("/content/recipe-agent/src")

import recipe_agent.tools as tools
import recipe_agent.agents as agents
importlib.reload(tools)
importlib.reload(agents)

print(tools._is_excluded("Easy Potato Gnocchi", ["potatoes", "gruyere cheese", "flour", "egg"], ["milk"]))
print(tools._is_excluded("Roast Chicken", ["chicken", "garlic", "lemon"], ["milk"]))
print(tools.compare_ingredients(
    ["potato", "egg"],
    ["boiling water", "table salt", "all-purpose flour", "instant mashed potatoes", "egg"],
))

True
False
{'have': ['egg'], 'missing': ['all-purpose flour', 'instant mashed potatoes'], 'coverage': 0.33}


In [30]:
def ask(text, top_k=3):
    req = agents.parse_request(text)
    found = tools.find_recipes(
        req.ingredients, max_minutes=req.max_minutes,
        exclude=req.exclude_ingredients, top_k=top_k,
    )
    for r in found:
        r["instructions"] = tools.get_instructions(r["recipe_id"])
    shop = tools.shopping_list([r["missing"] for r in found])
    print(req.model_dump(), "\n")
    print(agents.write_answer(req, found, shop))

ask("عندي بطاطس وبيض وعندي حساسية من اللبن")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'ingredients': ['potato', 'egg'], 'exclude_ingredients': ['milk'], 'max_minutes': None, 'language': 'ar'} 

ملاحظة: تصفية المكونات تعتمد على أسماء المكونات فقط، يرجى مراجعة الملصقات للتأكد من عدم وجود مواد تسبب الحساسية.

### نيوكي (Gnocchi)

- **الوقت الإجمالي:** 60 دقيقة
- **عدد الحصص:** 4
- **القيمة الغذائية (لكل حصة):** 327 سعرة حرارية، 10.2 غرام بروتين
- **المكونات المتوفرة لديك:** البطاطس (potatoes)، البيض (egg)
- **المكونات الناقصة:** دقيق لجميع الأغراض (all-purpose flour)

**الخطوات:**
1. اغلي قدرًا كبيرًا من الماء المملح. قشر البطاطس وأضفها إلى القدر. اطبخها حتى تصبح طرية ولكن لا تزال متماسكة، حوالي 15 دقيقة. صفها، واتركها تبرد، ثم اهرسها بشوكة أو مهرسة بطاطس.
2. اخلط كوبًا واحدًا من البطاطس المهروسة والدقيق والبيض في وعاء كبير. اعجن حتى تشكل العجينة كرة. شكل أجزاء صغيرة من العجينة على هيئة "أفاعي" طويلة. على سطح مرشوش بالدقيق، قطّع الأفاعي إلى قطع بحجم نصف بوصة.
3. اغلي قدرًا كبيرًا من الماء المملح خفيفًا. ألقِ النيوكي واطبخها لمدة 3 إلى 5 دقائق أو حتى تطفو على السطح؛ صفها و

In [31]:
%cd /content/recipe-agent
!git pull

/content/recipe-agent
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.86 KiB | 731.00 KiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   9c77f71..ced6086  main       -> origin/main
Updating 9c77f71..ced6086
Fast-forward
 src/recipe_agent/agents.py |  5 ++++-
 src/recipe_agent/tools.py  | 22 +++++++++++++++++-----
 2 files changed, 21 insertions(+), 6 deletions(-)


In [32]:
import os, sys, importlib
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"
sys.path.append("/content/recipe-agent/src")

import recipe_agent.tools as tools
import recipe_agent.agents as agents
importlib.reload(tools)
importlib.reload(agents)

steps_with_cream = ["Fry the patties.", "Serve with apple sauce and/or sour cream."]
steps_plain = ["Fry the patties.", "Serve with apple sauce."]
ings = ["potatoes", "eggs", "onion"]

print(tools._is_excluded("Potato Latkes", ings, ["milk"], steps_with_cream))
print(tools._is_excluded("Potato Latkes", ings, ["milk"], steps_plain))
print(tools._is_excluded("Potato Latkes", ings, ["milk"]))

True
False
False


In [33]:
def ask(text, top_k=3):
    req = agents.parse_request(text)
    found = tools.find_recipes(
        req.ingredients, max_minutes=req.max_minutes,
        exclude=req.exclude_ingredients, top_k=top_k,
    )
    for r in found:
        r["instructions"] = tools.get_instructions(r["recipe_id"])
    shop = tools.shopping_list([r["missing"] for r in found])
    print(req.model_dump(), "\n")
    print(agents.write_answer(req, found, shop))

ask("عندي بطاطس وبيض وعندي حساسية من اللبن")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'ingredients': ['potato', 'egg'], 'exclude_ingredients': ['milk'], 'max_minutes': None, 'language': 'ar'} 

ملاحظة: التصفية تستند إلى أسماء المكونات وخطوات الوصفة فقط، ويجب على المستخدم التحقق بنفسه من ملصقات المنتجات لتجنب الحساسية.

### نوكي (Gnocchi)

- **الوقت الإجمالي:** 60 دقيقة
- **عدد الحصص:** 4
- **السعرات الحرارية:** 327 سعرة حرارية لكل حصة
- **البروتين:** 10.2 جرام لكل حصة
- **المكونات المتوفرة لديك:** بطاطس (potatoes)، بيض (egg)
- **المكونات الناقصة:** دقيق متعدد الاستخدامات (all-purpose flour)

**الخطوات:**
1. اغلِ قدرًا كبيرًا من الماء المملح. قشر البطاطس وأضفها إلى القدر. اطبخها حتى تطرى ولكن تظل متماسكة، حوالي 15 دقيقة. صفِّها، واتركها تبرد واهرسها باستخدام شوكة أو هراسة بطاطس.
2. اخلط 1 كوب من البطاطس المهروسة والدقيق والبيض في وعاء كبير. اعجن حتى تتكون العجينة على شكل كرة. شكّل أجزاءً صغيرة من العجين على شكل "أفاعٍ" طويلة. على سطح مرشوش بالدقيق، اقطع الأفاعي إلى قطع بحجم نصف بوصة (half-inch).
3. اغلِ قدرًا كبيرًا من الماء المملح خفيفًا. ألقِ النوكي واطبخها لمدة 3 إلى

In [34]:
%cd /content/recipe-agent
!git pull

/content/recipe-agent
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.14 KiB | 1.07 MiB/s, done.
From https://github.com/Mennaali30/recipe-agent
   ced6086..40393cb  main       -> origin/main
Updating ced6086..40393cb
Fast-forward
 src/recipe_agent/graph.py | 88 +++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 88 insertions(+)
 create mode 100644 src/recipe_agent/graph.py


In [35]:
import os, sys, importlib
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"
sys.path.append("/content/recipe-agent/src")

import recipe_agent.tools, recipe_agent.agents
import recipe_agent.graph as graph
importlib.reload(recipe_agent.tools)
importlib.reload(recipe_agent.agents)
importlib.reload(graph)

print(graph.build_graph().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	search(search)
	clarify(clarify)
	respond(respond)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	parse -.-> clarify;
	parse -.-> search;
	search -.-> respond;
	clarify --> __end__;
	respond --> __end__;
	search -.-> search;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [36]:
def show(text):
    out = graph.run_agent(text)
    print("INPUT:", text)
    print("request:", out["request"].model_dump())
    print("search attempts:", out.get("attempt"), "| recipes:", len(out.get("recipes", [])))
    print(out["answer"])
    print("\n" + "=" * 70 + "\n")

show("عندي بطاطس وبيض وعندي حساسية من اللبن")
show("إزيك عامل إيه")
show("I have chicken, rice and garlic. Something quick, under 30 minutes.")

INPUT: عندي بطاطس وبيض وعندي حساسية من اللبن
request: {'ingredients': ['potato', 'egg'], 'exclude_ingredients': ['milk'], 'max_minutes': None, 'language': 'ar'}
search attempts: 2 | recipes: 3
ملاحظة: تصفية الوصفات تستند فقط إلى أسماء المكونات وخطوات التحضير، ويجب على المستخدم التحقق من الملصقات الغذائية للتأكد من الحساسية.

### نيوشي (Gnocchi)
- **الوقت الإجمالي:** 60 دقيقة
- **عدد الحصص:** 4
- **السعرات الحرارية:** 327 سعرة حرارية لكل حصة
- **البروتين:** 10.2 غرام لكل حصة
- **المكونات المتوفرة لديك:** بطاطس، بيض
- **المكونات المفقودة:** دقيق متعدد الاستخدامات (all-purpose flour)

**الخطوات:**
1. اغلِ قدرًا كبيرًا من الماء المملح. قشّر البطاطس وأضفها إلى القدر. اطبخها حتى تطرى ولكن تظل متماسكة، حوالي 15 دقيقة. صفّها، ودعها تبرد ثم اهرسها بشوكة أو بهراسة البطاطس.
2. اخلط كوبًا واحدًا من البطاطس المهروسة، والدقيق والبيضة في وعاء كبير. اعجن حتى تتكون العجينة على شكل كرة. شكّل أجزاء صغيرة من العجينة على شكل "ثعابين" طويلة. على سطح مرشوش بالدقيق، قطّع الثعابين إلى قطع بطول نصف بوصة.
3. اغل